# Implementasi Webinar Pertemuan 2
## Bridging Global Infrastructure to Address Local Challenges in High Performance Computing (HPC)

**Narasumber:** Dr. Warawan Dias Carbalo — Thammasat University, Thailand  
**Platform:** Python (Google Colab / Jupyter Notebook)

### Studi kasus baru: Simulasi Sederhana Sebaran Polusi Udara di Wilayah Perkotaan

Notebook ini mempertahankan tema webinar, yaitu menghubungkan infrastruktur komputasi berkinerja tinggi (*High Performance Computing/HPC*) dengan persoalan lokal. Studi kasus diubah dari simulasi potensi genangan menjadi simulasi sebaran konsentrasi polutan udara pada wilayah perkotaan hipotetis.

Tujuan pembelajaran:
1. Merumuskan masalah lokal menjadi persoalan komputasi terukur.
2. Membuat dan memvisualisasikan data spasial sintetis.
3. Membandingkan skenario emisi dan kondisi angin.
4. Mengukur waktu eksekusi dan memahami komputasi vektor.
5. Menjalankan banyak skenario dan merangkum variasi hasil.
6. Memahami alur kerja serta contoh skrip SLURM untuk klaster HPC.

> **Penting:** Model ini hanya untuk pembelajaran, bukan model dispersi atmosfer ilmiah. Data dan hasil sintetis tidak boleh digunakan untuk menilai kualitas udara nyata atau mengambil keputusan kesehatan masyarakat.

## 1. Memuat pustaka

NumPy digunakan untuk komputasi numerik, Pandas untuk merangkum hasil, Matplotlib untuk visualisasi, dan `time` untuk mengukur waktu eksekusi.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)
print("Pustaka berhasil dimuat.")

## 2. Merumuskan masalah lokal menjadi masalah komputasi

Pertanyaan studi kasus: **bagaimana perubahan intensitas emisi dan kondisi angin memengaruhi pola konsentrasi polutan dalam wilayah hipotetis?**

Wilayah direpresentasikan sebagai grid 2D. Sumber emisi menyumbang konsentrasi relatif berdasarkan jarak, sementara parameter angin digambarkan secara sederhana melalui pergeseran pusat sebaran. Model ini tidak mensimulasikan dinamika atmosfer secara lengkap. Studi nyata memerlukan data polutan dan meteorologi terukur, topografi, reaksi kimia, serta kalibrasi dan validasi.

## 3. Membuat grid wilayah dan sumber emisi sintetis

Grid 120 × 120 mewakili ruang hipotetis. Sumber emisi dibayangkan sebagai jalan utama atau kawasan industri, tetapi lokasinya bukan lokasi nyata.

In [ ]:
grid_size = 120
x = np.linspace(0, 10, grid_size)
y = np.linspace(0, 10, grid_size)
X, Y = np.meshgrid(x, y)

sources = [
    {"name": "Sumber A", "x": 3.0, "y": 3.5, "strength": 80.0},
    {"name": "Sumber B", "x": 6.5, "y": 7.0, "strength": 60.0},
    {"name": "Sumber C", "x": 8.0, "y": 2.5, "strength": 45.0},
]
sources_df = pd.DataFrame(sources)
display(sources_df)
print("Ukuran grid:", X.shape)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter([s["x"] for s in sources], [s["y"] for s in sources],
            s=[s["strength"] * 3 for s in sources], marker="x")
for s in sources:
    plt.annotate(s["name"], (s["x"], s["y"]), xytext=(5, 5), textcoords="offset points")
plt.title("Lokasi Sumber Emisi pada Wilayah Hipotetis")
plt.xlabel("Koordinat X"); plt.ylabel("Koordinat Y")
plt.xlim(0, 10); plt.ylim(0, 10); plt.grid(alpha=0.25)
plt.show()

## 4. Membuat fungsi simulasi konsentrasi relatif

Setiap sumber menyumbang konsentrasi relatif berdasarkan kekuatan emisi dan jarak. Pengaruh angin disederhanakan sebagai pergeseran pusat sebaran, sedangkan `spread` mengatur seberapa cepat pengaruh sumber menyebar.

Nilai keluaran bukan satuan PM2.5 atau konsentrasi terukur; ini adalah **indeks relatif sintetis** untuk membandingkan skenario.

In [ ]:
def simulate_pollution(X, Y, sources, emission_scale=1.0,
                       wind_x=0.0, wind_y=0.0, spread=0.8):
    """Indeks konsentrasi relatif sintetis, bukan prediksi kualitas udara nyata."""
    field = np.zeros_like(X, dtype=float)
    for source in sources:
        dx = X - (source["x"] + wind_x)
        dy = Y - (source["y"] + wind_y)
        distance_sq = dx**2 + dy**2
        field += (source["strength"] * emission_scale) * np.exp(
            -distance_sq / (2 * spread**2)
        )
    return field

base_field = simulate_pollution(X, Y, sources, emission_scale=1.0,
                                wind_x=0.3, wind_y=0.1, spread=0.8)
print(f"Indeks minimum: {base_field.min():.3f}")
print(f"Indeks maksimum: {base_field.max():.3f}")
print(f"Indeks rata-rata: {base_field.mean():.3f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(base_field, origin="lower", extent=[0, 10, 0, 10], aspect="auto")
plt.colorbar(label="Indeks konsentrasi relatif (sintetis)")
plt.scatter([s["x"] for s in sources], [s["y"] for s in sources], marker="x", label="Sumber emisi")
plt.title("Simulasi Sebaran Polutan — Skenario Dasar")
plt.xlabel("Koordinat X"); plt.ylabel("Koordinat Y"); plt.legend()
plt.show()

## 5. Membandingkan skenario emisi dan angin

Kita mengubah skala emisi dan pergeseran angin, kemudian menghitung indeks maksimum, rata-rata, dan jumlah sel di atas ambang ilustratif. Ambang ini hanya untuk perbandingan internal, bukan ambang aman atau berbahaya bagi kesehatan.

In [ ]:
scenarios = [
    {"scenario": "Emisi rendah", "emission_scale": 0.6, "wind_x": 0.3, "wind_y": 0.1},
    {"scenario": "Kondisi dasar", "emission_scale": 1.0, "wind_x": 0.3, "wind_y": 0.1},
    {"scenario": "Emisi tinggi", "emission_scale": 1.5, "wind_x": 0.3, "wind_y": 0.1},
    {"scenario": "Angin ke timur", "emission_scale": 1.0, "wind_x": 1.0, "wind_y": 0.0},
    {"scenario": "Angin ke utara", "emission_scale": 1.0, "wind_x": 0.0, "wind_y": 1.0},
]
illustrative_threshold = 8.0
scenario_rows, scenario_fields = [], {}
for sc in scenarios:
    field = simulate_pollution(X, Y, sources, emission_scale=sc["emission_scale"],
                               wind_x=sc["wind_x"], wind_y=sc["wind_y"])
    scenario_fields[sc["scenario"]] = field
    scenario_rows.append({
        "skenario": sc["scenario"],
        "indeks_maksimum": field.max(),
        "indeks_rata_rata": field.mean(),
        "sel_di_atas_ambang": int((field >= illustrative_threshold).sum()),
        "persentase_sel_di_atas_ambang": (field >= illustrative_threshold).mean() * 100
    })
scenario_results = pd.DataFrame(scenario_rows)
display(scenario_results.round(2))

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(scenario_results["skenario"], scenario_results["indeks_maksimum"])
plt.title("Perbandingan Indeks Maksimum per Skenario")
plt.ylabel("Indeks maksimum (sintetis)")
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ["Emisi rendah", "Kondisi dasar", "Emisi tinggi"]):
    im = ax.imshow(scenario_fields[name], origin="lower", extent=[0, 10, 0, 10], aspect="auto")
    ax.set_title(name); ax.set_xlabel("Koordinat X"); ax.set_ylabel("Koordinat Y")
fig.colorbar(im, ax=axes.ravel().tolist(), label="Indeks relatif", shrink=0.8)
plt.show()

### Interpretasi

Perubahan skala emisi mengubah intensitas indeks dalam model. Pergeseran angin mengubah lokasi relatif area dengan nilai lebih tinggi. Interpretasi ini hanya berlaku dalam asumsi persamaan sintetis dan tidak dapat langsung diterapkan pada wilayah nyata.

## 6. Mengapa HPC dapat dibutuhkan?

HPC relevan ketika resolusi grid, jumlah sumber, kompleksitas model, jumlah langkah waktu, atau jumlah skenario meningkat. Grid 120 × 120 memiliki 14.400 sel, sedangkan grid 2.000 × 2.000 memiliki 4 juta sel.

Sebagai demonstrasi, kita membandingkan loop Python dengan operasi array NumPy. Ini bukan benchmark klaster HPC, tetapi menunjukkan bagaimana pilihan implementasi memengaruhi efisiensi.

In [ ]:
benchmark_size = 500_000
values = rng.random(benchmark_size)

start = time.perf_counter()
loop_result = np.empty(benchmark_size)
for i in range(benchmark_size):
    loop_result[i] = values[i] * 1.25 + 0.5
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
vector_result = values * 1.25 + 0.5
vector_seconds = time.perf_counter() - start

print(f"Waktu loop Python : {loop_seconds:.6f} detik")
print(f"Waktu NumPy vektor: {vector_seconds:.6f} detik")
if vector_seconds > 0:
    print(f"Rasio waktu loop / vektor: {loop_seconds / vector_seconds:.2f}x")
print("Hasil sama:", np.allclose(loop_result, vector_result))

### Catatan benchmark

Waktu eksekusi bergantung pada perangkat, versi pustaka, dan beban sistem. Komputasi vektor dapat mengurangi overhead loop Python, tetapi tidak berarti semua pekerjaan otomatis lebih cepat. Performa HPC juga dipengaruhi pembagian pekerjaan, komunikasi antarproses, memori, penyimpanan, dan ukuran masalah.

## 7. Simulasi Monte Carlo untuk variasi parameter

Kita membuat banyak skenario dengan variasi acak pada skala emisi dan pergeseran angin. Tujuannya menunjukkan bagaimana simulasi berulang dapat merangkum rentang keluaran, bukan memprediksi kondisi nyata. Pada skala besar, skenario independen dapat dibagi ke beberapa proses atau node HPC.

In [ ]:
n_simulations = 1500
emission_samples = rng.uniform(0.5, 1.6, n_simulations)
wind_x_samples = rng.uniform(-0.5, 1.0, n_simulations)
wind_y_samples = rng.uniform(-0.5, 1.0, n_simulations)

max_index_samples = np.empty(n_simulations)
mean_index_samples = np.empty(n_simulations)
above_threshold_samples = np.empty(n_simulations)

for i in range(n_simulations):
    field = simulate_pollution(X, Y, sources, emission_scale=emission_samples[i],
                               wind_x=wind_x_samples[i], wind_y=wind_y_samples[i])
    max_index_samples[i] = field.max()
    mean_index_samples[i] = field.mean()
    above_threshold_samples[i] = (field >= illustrative_threshold).mean() * 100

mc_results = pd.DataFrame({
    "emission_scale": emission_samples, "wind_x": wind_x_samples, "wind_y": wind_y_samples,
    "max_index": max_index_samples, "mean_index": mean_index_samples,
    "percent_cells_above_threshold": above_threshold_samples
})
display(mc_results[["max_index", "mean_index", "percent_cells_above_threshold"]]
        .describe(percentiles=[0.05, 0.50, 0.95]).round(3))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(max_index_samples, bins=30)
plt.title("Distribusi Indeks Maksimum dari Simulasi Monte Carlo")
plt.xlabel("Indeks maksimum (sintetis)"); plt.ylabel("Frekuensi"); plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(emission_samples, max_index_samples, alpha=0.35)
plt.title("Skala Emisi dan Indeks Maksimum")
plt.xlabel("Skala emisi"); plt.ylabel("Indeks maksimum"); plt.show()

## 8. Contoh alur kerja pada klaster HPC

Penggunaan HPC bukan hanya menulis kode. Pengguna perlu memahami akses akun, lingkungan Linux, terminal/SSH atau portal yang disediakan, penyimpanan, batas sumber daya, sistem antrean, log, dan dokumentasi.

Alur umum:
1. Menyiapkan kode dan data.
2. Mengakses fasilitas dengan akun resmi.
3. Menyiapkan lingkungan dan dependensi.
4. Menulis skrip pekerjaan serta menentukan kebutuhan CPU, memori, dan waktu.
5. Mengirim pekerjaan ke antrean.
6. Memantau status dan membaca log.
7. Memeriksa hasil serta mendokumentasikan konfigurasi agar eksperimen dapat diulang.

Perintah aktual harus mengikuti dokumentasi fasilitas HPC.

### Contoh perintah terminal (ilustrasi)

```bash
# Gunakan alamat dan akun yang diberikan administrator fasilitas
ssh username@alamat-klaster
pwd
ls -lh

sbatch pollution_job.slurm
squeue -u "$USER"
cat pollution-job-<JOB_ID>.out
cat pollution-job-<JOB_ID>.err
```

Alamat dan nama pengguna di atas hanya contoh. Notebook ini tidak terhubung ke klaster HPC.

## 9. Contoh skrip SLURM untuk menjalankan Python

Skrip berikut menunjukkan bagaimana pekerjaan Python dapat dijadwalkan pada fasilitas yang menggunakan SLURM. Nama partisi, modul Python, batas waktu, dan kebutuhan memori harus disesuaikan. Notebook ini hanya menampilkan dan menyimpan contoh skrip, bukan mengirim pekerjaan ke klaster.

In [ ]:
slurm_script = """#!/bin/bash
#SBATCH --job-name=pollution-simulation
#SBATCH --output=pollution-job-%j.out
#SBATCH --error=pollution-job-%j.err
#SBATCH --time=00:10:00
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G

# Sesuaikan lingkungan Python dengan panduan fasilitas.
# module load python

python pollution_simulation.py
"""
print(slurm_script)

In [ ]:
with open("pollution_job.slurm", "w", encoding="utf-8") as f:
    f.write(slurm_script)
print("Contoh skrip disimpan sebagai pollution_job.slurm")

## 10. Menghubungkan simulasi dengan kebutuhan lokal

Simulasi polusi udara menunjukkan cara menjembatani infrastruktur komputasi dengan masalah lokal. Aplikasi nyata memerlukan kolaborasi antara peneliti komputasi, ahli lingkungan, instansi pemilik data, pemerintah, dan masyarakat.

Data terukur dan model tervalidasi dapat mendukung kajian lingkungan, tetapi indeks sintetis dalam notebook ini bukan pengganti sensor kualitas udara atau pedoman kesehatan.

## 11. Keterampilan yang perlu dikembangkan

1. Python dan pemrograman ilmiah.
2. Linux, terminal, direktori, dan pengelolaan berkas.
3. Pengolahan dan pemeriksaan kualitas data.
4. Pengukuran performa dan optimasi kode.
5. Dasar paralelisasi dan pembagian pekerjaan.
6. Sistem antrean seperti SLURM sesuai fasilitas.
7. Dokumentasi, reproduksibilitas, keamanan data, dan kolaborasi lintas disiplin.

Kode yang berjalan di laptop belum tentu langsung berjalan di klaster; lingkungan, dependensi, lokasi data, dan batas sumber daya perlu disesuaikan.

## 12. Kesimpulan

Studi kasus ini memperlihatkan cara menerjemahkan masalah lingkungan lokal menjadi simulasi komputasi, membandingkan skenario, memvisualisasikan hasil, serta mengeksplorasi variasi parameter melalui simulasi berulang. Contoh benchmark memperkenalkan manfaat operasi vektor, sementara contoh SLURM menggambarkan salah satu bagian alur kerja klaster HPC.

HPC membantu menangani masalah dengan resolusi dan jumlah simulasi yang lebih besar, tetapi hasil yang berguna tetap bergantung pada kualitas data, validitas model, pemahaman bidang, kolaborasi, dan pengelolaan pekerjaan yang benar.

### Keterbatasan implementasi
- Semua data, sumber emisi, dan hasil bersifat sintetis.
- Persamaan dispersi sangat sederhana dan bukan model atmosfer tervalidasi.
- Notebook berjalan di lingkungan Python biasa dan tidak mengakses klaster HPC.
- Contoh SLURM hanya ilustrasi dan perlu disesuaikan dengan fasilitas.
- Hasil tidak boleh digunakan untuk prediksi kualitas udara nyata, keputusan kesehatan, atau kebijakan lingkungan.